# Primary judge inference: Qwen2.5-7B-Instruct (RQ1–RQ5)

Colab (L4) record of every GPU run for the primary judge. Outputs are kept as evidence of the real runs.

- `src.judge --condition clean`: P1/P2/P3, both orders, greedy + `k_sc` sampled calls on P1
- `src.judge --condition verbose`: P1, both orders, greedy only
- `src.vacuum_test`: identical/empty-pair probe (task 1.8)
- `src.ablation_decoding`: constrained vs free-form decoding (task 4.5)

Checkpoints land in `runs/qwen2.5_7b_instruct/` on Drive and are copied back locally (D17). Parsing, signals and all analysis run locally, see `README.md`. Some smoke-test cells predate D26 and use the old unsuffixed paths (`runs/judge_clean.jsonl`).

**Step 1: Mount Google Drive where the project will be cloned**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


**Step 2: For first time, clone the repo**

In [ ]:
import getpass
import os

# Securely prompt for GitHub Personal Access Token
token = getpass.getpass('Enter your GitHub PAT: ')
repo_url = "https://" + token + "@github.com/senguptashubham/judge-calibration.git"
dest_path = "/content/drive/MyDrive/judge-calibration"

!git clone {repo_url} {dest_path}

Enter your GitHub PAT: ··········
Cloning into '/content/drive/MyDrive/judge-calibration'...
remote: Enumerating objects: 162, done.
remote: Counting objects: 100% (162/162), done.
remote: Compressing objects: 100% (114/114), done.
remote: Total 162 (delta 89), reused 119 (delta 46), pack-reused 0 (from 0)
Receiving objects: 100% (162/162), 136.08 KiB | 8.50 MiB/s, done.
Resolving deltas: 100% (89/89), done.


**Step 3: Next time onwards point to repo and pull latest changes**

In [ ]:
import getpass
!cd /content/drive/MyDrive/judge-calibration && git remote set-url origin https://github.com/senguptashubham/judge-calibration.git
token = getpass.getpass('Enter your GitHub PAT: ')
!cd /content/drive/MyDrive/judge-calibration && git pull https://{token}@github.com/senguptashubham/judge-calibration.git main


Enter your GitHub PAT: ··········
remote: Enumerating objects: 200, done.
remote: Counting objects: 100% (127/127), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 200 (delta 95), reused 80 (delta 48), pack-reused 73 (from 1)
Receiving objects: 100% (200/200), 283.95 KiB | 49.00 KiB/s, done.
Resolving deltas: 100% (140/140), completed with 8 local objects.
From https://github.com/senguptashubham/judge-calibration
 * branch            main       -> FETCH_HEAD
Updating 62201eb..6f8a2a6
Fast-forward
 CLAUDE.md               |   12 +-
 DECISIONS.md            |   41 +-
 PLAN.md                 |    6 +-
 REPORT.md               |  554 +++++++++++++++++
 TASKS.md                |  122 +++-
 analysis/rq4.py         | 1576 +++++++++++++++++++++++++++++++++++++++++++++++
 analysis/rq5.py         |  348 ++++++++++-
 configs/run_kev.yaml    |   39 ++
 src/bayesian.py         |  695 +++++++++++++++++++++
 src/data.py             |   28 +-
 src/features.py         |  136 +++-


**Step 4: Install uv, create venv and set path, install dependencies**

In [ ]:
import os

%cd /content/drive/MyDrive/judge-calibration

!pip install -q uv
!uv venv /content/venv --python 3.11 --clear
!uv pip install --python /content/venv/bin/python -e ".[colab]"

os.environ["PATH"] = "/content/venv/bin:" + os.environ.get("PATH", "")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/judge-calibration
Using CPython 3.11.16
Creating virtual environment at: /content/venv
Activate with: source /content/venv/bin/activate
Using Python 3.11.16 environment at: /content/venv
Resolved 227 packages in 2.64s
Prepared 172 packages in 4m 47s
Installed 227 packages in 315ms
 + agent-detector==2.0.0
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.3
 + aiosignal==1.4.0
 + annotated-doc==0.0.5
 + annotated-types==0.8.0
 + anthropic==1.7.0
 + anyio==4.15.1
 + apache-tvm-ffi==0.1.11
 + arviz==0.23.4
 + astor==0.8.1
 + attrs==26.1.0
 + blake3==1.0.9
 + cachetools==7.2.0
 + cbor2==6.1.4
 + certifi==2026.7.22
 + cffi==2.1.1
 + charset-normalizer==3.5.1
 + click==8.5.0
 + cloudpickle==3.1.2
 + compressed-tensors==0.17.0
 + contourpy==1.3.3
 + cryptography==50.0.1
 + cuda-bindings==13.4.2
 + cuda-core==1.2.0
 + cuda-pathfinder==1.8.2
 + cuda-p

**Step 5: check supported functionality before use**

In [ ]:
!/content/venv/bin/python -c "from vllm.sampling_params import StructuredOutputsParams; help(StructuredOutputsParams)"


Help on class StructuredOutputsParams in module vllm.sampling_params:

class SSttrruuccttuurreeddOOuuttppuuttssPPaarraammss(builtins.object)
 |  StructuredOutputsParams(json: str | dict | None = None, regex: str | None = None, choice: list[str] | None = None, grammar: str | None = None, json_object: bool | None = None, disable_any_whitespace: bool = False, disable_additional_properties: bool = False, whitespace_pattern: str | None = None, structural_tag: str | None = None) -> None
 |  
 |  # maybe make msgspec?
 |  
 |  Methods defined here:
 |  
 |  ____eeqq____(self, other)
 |      Return self==value.
 |  
 |  ____iinniitt____(__dataclass_self__: 'PydanticDataclass', *args: 'Any', **kwargs: 'Any') -> 'None' from pydantic._internal._dataclasses.StructuredOutputsParams
 |      # dataclass.__init__ must be defined here so its `__qualname__` can be changed since functions can't be copied,
 |      # and so that the mock validator is used if building wa

**Optional: If ninja is not installed, install it**

In [ ]:
!uv pip install --python /content/venv/bin/python ninja


Using Python 3.11.16 environment at: /content/venv
Checked 1 package in 4ms


**Step 6: Run smoke test on selective items** (optional)

In [ ]:
!/content/venv/bin/python -m src.judge --config configs/run.yaml --condition clean --n-items 20 --prompt-variants P1


INFO 09-08 11:56:34 [api_utils.py:272] non-default args: {'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-7B-Instruct'}
INFO 09-08 11:56:35 [model.py:672] Resolved architecture: Qwen2ForCausalLM
INFO 09-08 11:56:35 [model.py:1965] Using max model len 32768
INFO 09-08 11:56:35 [scheduler.py:242] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 09-08 11:56:35 [kernel.py:308] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
(EngineCore pid=14264) INFO 09-08 11:56:39 [core.py:122] Initializing a V1 LLM engine (v0.28.0) with config: model='Qwen/Qwen2.5-7B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-7B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=main, tokenizer_revision=main, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=32768, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_cont

**Step 7: Check the written json** (optional)

In [ ]:
!wc -l runs/judge_clean.jsonl
!ls runs/logprobs | wc -l


120 runs/judge_clean.jsonl
120


**Step 8: Run second smoke test on selective items** (optional)

In [ ]:
!/content/venv/bin/python -m src.judge --config configs/run.yaml --condition clean --n-items 5 --prompt-variants P2,P3


remote: Enumerating objects: 8, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 6 (delta 3), reused 6 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 3.31 KiB | 45.00 KiB/s, done.
From https://github.com/senguptashubham/judge-calibration
 * branch            main       -> FETCH_HEAD
Updating a58c4cb..3851b3a
Fast-forward
 REPORT.md      | 63 ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
 pyproject.toml |  7 +++++++
 2 files changed, 70 insertions(+)
 create mode 100644 REPORT.md
INFO 09-08 12:41:25 [api_utils.py:272] non-default args: {'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-7B-Instruct'}
INFO 09-08 12:41:27 [model.py:672] Resolved architecture: Qwen2ForCausalLM
INFO 09-08 12:41:27 [model.py:1965] Using max model len 32768
INFO 09-08 12:41:27 [scheduler.py:242] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 09-08 12:41:27 [kernel.py:308] Final IR op priority after s

**Step 9: Run vacuum test on full data**

In [ ]:
!/content/venv/bin/python -m src.vacuum_test --config configs/run.yaml


INFO 09-10 10:29:28 [api_utils.py:272] non-default args: {'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-7B-Instruct'}
INFO 09-10 10:29:29 [model.py:672] Resolved architecture: Qwen2ForCausalLM
INFO 09-10 10:29:29 [model.py:1965] Using max model len 32768
INFO 09-10 10:29:29 [scheduler.py:242] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 09-10 10:29:29 [kernel.py:308] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
(EngineCore pid=9023) INFO 09-10 10:29:33 [core.py:122] Initializing a V1 LLM engine (v0.28.0) with config: model='Qwen/Qwen2.5-7B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-7B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=main, tokenizer_revision=main, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=32768, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_conte

**Step 10: Run clean condition test on full data**

In [ ]:
!/content/venv/bin/python -m src.judge --config configs/run.yaml --condition clean








INFO 09-14 10:32:08 [api_utils.py:272] non-default args: {'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-7B-Instruct'}

INFO 09-14 10:32:22 [model.py:672] Resolved architecture: Qwen2ForCausalLM
INFO 09-14 10:32:22 [model.py:1965] Using max model len 32768
INFO 09-14 10:32:22 [scheduler.py:242] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 09-14 10:32:22 [kernel.py:308] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])





(EngineCore pid=6375) INFO 09-14 10:32:28 [core.py:122] Initializing a V1 LLM engine (v0.28.0) with config: model='Qwen/Qwen2.5-7B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-7B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=main, tokenizer_revision=main, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=32768, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1,

**Step 11: Run verbose condition test on full data**

In [ ]:
!/content/venv/bin/python -m src.judge --config configs/run.yaml --condition verbose


INFO 09-18 06:57:57 [api_utils.py:272] non-default args: {'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-7B-Instruct'}
INFO 09-18 06:57:58 [model.py:672] Resolved architecture: Qwen2ForCausalLM
INFO 09-18 06:57:58 [model.py:1965] Using max model len 32768
INFO 09-18 06:57:58 [scheduler.py:242] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 09-18 06:57:58 [kernel.py:308] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
(EngineCore pid=11069) INFO 09-18 06:58:02 [core.py:122] Initializing a V1 LLM engine (v0.28.0) with config: model='Qwen/Qwen2.5-7B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-7B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=main, tokenizer_revision=main, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=32768, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_cont

**Step 12: check time and generated json after test**

In [ ]:
import json

rows = []
with open("/content/drive/MyDrive/judge-calibration/runs/qwen2.5_7b_instruct/judge_verbose.jsonl") as f:
    for line in f:
        rows.append(json.loads(line))

latencies_sec = [r["latency_ms"] / 1000 for r in rows if r["condition"] == "verbose"]
n = len(latencies_sec)
mean_rate = sum(latencies_sec) / n

print(f"n generations: {n}")
print(f"mean verbose rate: {mean_rate:.3f} sec/gen")
print(f"clean rate (measured earlier, L4): 1.200 sec/gen")
print(f"naive token-multiplier estimate: 3.600 sec/gen")
print(f"real multiplier vs clean: {mean_rate / 1.2:.3f}x")


n generations: 3808
mean verbose rate: 0.664 sec/gen
clean rate (measured earlier, L4): 1.200 sec/gen
naive token-multiplier estimate: 3.600 sec/gen
real multiplier vs clean: 0.554x


In [ ]:
import json
rows = [json.loads(l) for l in open("/content/drive/MyDrive/judge-calibration/runs/qwen2.5_7b_instruct/judge_verbose.jsonl")]
mean_prompt_tokens = sum(r["n_prompt_tokens"] for r in rows) / len(rows)
print(mean_prompt_tokens)  # expect ~2450, not ~980


2449.5


**Step 13: run ablation decoding test on few items**

In [ ]:
!/content/venv/bin/python -m src.ablation_decoding --config configs/run.yaml --n-items 100









INFO 09-18 10:08:38 [api_utils.py:272] non-default args: {'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-7B-Instruct'}

INFO 09-18 10:08:51 [model.py:672] Resolved architecture: Qwen2ForCausalLM
INFO 09-18 10:08:51 [model.py:1965] Using max model len 32768
INFO 09-18 10:08:51 [scheduler.py:242] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 09-18 10:08:51 [kernel.py:308] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])





(EngineCore pid=4955) INFO 09-18 10:08:58 [core.py:122] Initializing a V1 LLM engine (v0.28.0) with config: model='Qwen/Qwen2.5-7B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-7B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=main, tokenizer_revision=main, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=32768, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1,